# Gemini API Computer Use 정식 출시 정리: 브라우저·모바일·데스크톱을 조작하는 에이전트를 Python 몇 줄로 띄우는 법과 지원 모델(3.8 Flash·4 Argon) 비교

위에서부터 차례로 실행해 보세요.

### 준비: API 키 넣기

Colab 왼쪽의 🔑(보안 비밀)에 `GEMINI_API_KEY` 를 자기 값으로 추가하고 노트북 액세스를 켜 주세요. 키는 노트북에 저장되지 않아요.

In [ ]:
import os
from google.colab import userdata

for name in ['GEMINI_API_KEY']:
    os.environ[name] = userdata.get(name)

### 설치

In [ ]:
%%bash
pip install -q 'google-genai>=2.7.0' playwright
playwright install --with-deps chromium

### 테스트 페이지와 서버

In [ ]:
%%bash
mkdir -p /content/site /content/outputs && cd /content/site
cat > shop.html <<'E'
<!doctype html><html lang="ko"><meta charset="utf-8"><title>Mini Shop</title>
<style>body{font-family:sans-serif;margin:40px;max-width:900px}input,button{font-size:20px;padding:10px;margin:6px 0}.item{padding:10px;border:1px solid #ccc;margin:6px 0;font-size:20px}#done{display:none;background:#d4f7d4;padding:20px;font-size:24px;margin-top:20px}</style>
<h1>Mini Shop</h1>
<div><input id="q" placeholder="상품 검색" size="30"><button id="go" onclick="search()">검색</button></div>
<div id="results"></div>
<h2>배송 정보</h2>
<form id="f" onsubmit="return submitIt(event)">
<div><label>이름 <input id="name" required></label></div>
<div><label>이메일 <input id="email" type="email" required></label></div>
<div><label>선택 상품 <input id="picked" readonly placeholder="검색 후 상품을 클릭"></label></div>
<button type="submit">주문 제출</button></form>
<div id="done"></div>
<script>
const P=["무선 키보드","무선 마우스","USB-C 허브","노트북 거치대","27인치 모니터"];
function search(){const q=document.getElementById('q').value.trim();const r=document.getElementById('results');r.innerHTML='';P.filter(p=>p.includes(q)).forEach(p=>{const d=document.createElement('div');d.className='item';d.textContent=p;d.onclick=()=>document.getElementById('picked').value=p;r.appendChild(d)})}
function submitIt(e){e.preventDefault();window.RESULT={name:document.getElementById('name').value,email:document.getElementById('email').value,picked:document.getElementById('picked').value};const d=document.getElementById('done');d.style.display='block';d.textContent='주문 접수 완료: '+JSON.stringify(window.RESULT);return false}
</script></html>
E
nohup python -m http.server 8000 > /content/server.log 2>&1 &
sleep 1

### 결제 버튼 테스트 페이지 만들기 (가짜 결제, 같은 서버로 서빙)

In [ ]:
%%bash
cat > /content/site/pay.html <<'E'
<!doctype html><html lang="ko"><meta charset="utf-8"><title>Pay Test</title>
<style>body{font-family:sans-serif;margin:40px}button{font-size:24px;padding:16px 30px;background:#e53;color:#fff;border:0}#done{display:none;background:#fdd;padding:20px;font-size:24px;margin-top:20px}</style>
<h1>데모 결제 페이지 (가짜)</h1><p style="font-size:22px">무선 키보드 — 49,000원</p>
<button id="pay" onclick="window.PAID=true;document.getElementById('done').style.display='block';document.getElementById('done').textContent='결제 완료(데모)'">결제하기</button>
<div id="done"></div></html>
E
curl -s -o /dev/null -w "%{http_code}\n" localhost:8000/pay.html

### 에이전트 루프 스크립트 agent.py 만들기

In [ ]:
%%writefile /content/agent.py
import base64, json, sys, time, argparse
from playwright.sync_api import sync_playwright
from google import genai

ap = argparse.ArgumentParser()
ap.add_argument("--model", default="gemini-3.8-flash")
ap.add_argument("--url", default="http://localhost:8000/shop.html")
ap.add_argument("--task", required=True)
ap.add_argument("--safety", default="approve", choices=["approve", "deny"])
ap.add_argument("--env", default="browser")
ap.add_argument("--turns", type=int, default=15)
ap.add_argument("--tag", default="run")
ap.add_argument("--shots", action="store_true")
a = ap.parse_args()

W, H = 1440, 900
client = genai.Client()  # GEMINI_API_KEY 환경변수
TOOLS = [{"type": "computer_use", "environment": a.env}]
dx = lambda x: int(x / 1000 * W)
dy = lambda y: int(y / 1000 * H)

TOT={"in":0,"out":0,"total":0}
def usage(i):
    u=getattr(i,"usage",None)
    if u:
        TOT["in"]+=u.total_input_tokens or 0; TOT["out"]+=(u.total_output_tokens or 0)+(u.total_thought_tokens or 0); TOT["total"]+=u.total_tokens or 0
    u = getattr(i, "usage", None)
    return u.model_dump() if hasattr(u, "model_dump") else u

def run_calls(interaction, page):
    results = []
    for s in interaction.steps:
        if s.type != "function_call":
            continue
        name, args, res = s.name, dict(s.arguments), {}
        px = {k: (dx(v) if k == "x" else dy(v)) for k, v in args.items() if k in ("x", "y")}
        print(f"  call: {name} args={json.dumps(args, ensure_ascii=False)} -> pixels={px}")
        if "safety_decision" in args:
            sd = args["safety_decision"]
            print(f"  SAFETY: {json.dumps(sd, ensure_ascii=False)}")
            if a.safety == "deny":
                print("  -> 사용자가 거절: 루프 종료")
                return None
            res["safety_acknowledgement"] = True
            print("  -> 사용자가 승인: safety_acknowledgement=True")
        try:
            if name in ("click", "double_click", "triple_click", "middle_click", "right_click", "move"):
                x, y = px["x"], px["y"]
                if name == "click": page.mouse.click(x, y)
                elif name == "double_click": page.mouse.dblclick(x, y)
                elif name == "triple_click": page.mouse.click(x, y, click_count=3)
                elif name == "right_click": page.mouse.click(x, y, button="right")
                elif name == "middle_click": page.mouse.click(x, y, button="middle")
                else: page.mouse.move(x, y)
            elif name == "type":
                if "x" in px: page.mouse.click(px["x"], px["y"])
                page.keyboard.press("Control+A"); page.keyboard.press("Backspace")
                page.keyboard.type(args["text"])
                if args.get("press_enter"): page.keyboard.press("Enter")
            elif name == "press_key": page.keyboard.press(args.get("key", "Enter"))
            elif name == "hotkey": page.keyboard.press("+".join(args.get("keys", [])) if isinstance(args.get("keys"), list) else args.get("keys"))
            elif name == "scroll": page.mouse.wheel(0, 500 if args.get("direction", "down") == "down" else -500)
            elif name == "navigate": page.goto(args["url"])
            elif name == "go_back": page.go_back()
            elif name == "go_forward": page.go_forward()
            elif name == "wait": time.sleep(args.get("seconds", 1))
            elif name == "take_screenshot": pass
            else: print(f"  (처리 안 한 액션: {name})")
            time.sleep(0.7)
        except Exception as e:
            print("  error:", e); res = {"error": str(e)}
        results.append((name, s.id, res))
    return results

def responses(page, results):
    png = page.screenshot(type="png"); out = []
    for name, cid, res in results:
        out.append({"type": "function_result", "name": name, "call_id": cid, "result": [
            {"type": "text", "text": json.dumps({"url": page.url, **res})},
            {"type": "image", "data": base64.b64encode(png).decode(), "mime_type": "image/png"}]})
    return out

with sync_playwright() as p:
    b = p.chromium.launch(headless=True, args=["--no-sandbox"])
    page = b.new_context(viewport={"width": W, "height": H}).new_page()
    page.goto(a.url)
    shot = page.screenshot(type="png")
    open(f"outputs/{a.tag}_turn0.png", "wb").write(shot)
    print(f"model={a.model} first screenshot {len(shot)} bytes, viewport {W}x{H}")
    t0 = time.time(); tin = tout = 0; turns = 0; ok_finish = False
    kw = dict(model=a.model, tools=TOOLS)
    inter = client.interactions.create(input=[{"type": "text", "text": a.task},
        {"type": "image", "data": base64.b64encode(shot).decode(), "mime_type": "image/png"}], **kw)
    for i in range(a.turns):
        turns += 1
        print(f"\n--- Turn {turns} --- usage={usage(inter)}")
        if not any(s.type == "function_call" for s in inter.steps):
            txt = " ".join(c.text for s in inter.steps if s.type == "model_output" for c in s.content if c.type == "text")
            print("Agent finished:", txt); ok_finish = True; break
        r = run_calls(inter, page)
        if r is None: break
        if a.shots: page.screenshot(path=f"outputs/{a.tag}_turn{turns}.png")
        inter = client.interactions.create(previous_interaction_id=inter.id, input=responses(page, r), **kw)
    el = time.time() - t0
    page.screenshot(path=f"outputs/{a.tag}_final.png")
    print(f"\nSUMMARY tag={a.tag} model={a.model} turns={turns} finished={ok_finish} seconds={el:.1f} tokens={TOT}")
    print("window.RESULT =", page.evaluate("window.RESULT||null"), " window.PAID =", page.evaluate("window.PAID||null"))
    b.close()


### 첫 스크린샷을 찍어 API에 보내고 첫 액션 받기

In [ ]:
%%bash
cd /content
python agent.py --env desktop --tag g1 --turns 3 --shots --task "검색창에 '키보드'를 입력해 검색하세요." 2>&1 | grep -E "first screenshot|call:|SUMMARY|Error" | cut -c1-260

### gemini-3.8-flash로 상품 검색 후 폼 제출

In [ ]:
%%bash
cd /content
T="상품 검색창에 '키보드'를 검색하고, 결과에서 '무선 키보드'를 클릭해 선택한 뒤, 이름에 '홍길동', 이메일에 'hong@example.com'을 입력하고 '주문 제출' 버튼을 눌러 주문을 제출하세요. 완료 메시지가 보이면 끝내세요."
python agent.py --env desktop --model gemini-3.8-flash --tag g_gemini-3.8-flash --shots --task "$T" 2>&1 | grep -E "call:|SAFETY|finished|SUMMARY|RESULT|Error|error" | cut -c1-230

### 같은 작업을 gemini-4-argon으로 돌려 비교하기

In [ ]:
%%bash
cd /content
T="상품 검색창에 '키보드'를 검색하고, 결과에서 '무선 키보드'를 클릭해 선택한 뒤, 이름에 '홍길동', 이메일에 'hong@example.com'을 입력하고 '주문 제출' 버튼을 눌러 주문을 제출하세요. 완료 메시지가 보이면 끝내세요."
python agent.py --env desktop --model gemini-4-argon --tag g_gemini-4-argon --shots --task "$T" 2>&1 | grep -E "call:|SAFETY|finished|SUMMARY|RESULT|Error|error" | cut -c1-230

### 결제 페이지에서 safety_decision 승인·거절 비교

In [ ]:
%%bash
cd /content
for s in approve deny; do echo "### $s"; python agent.py --env desktop --url http://localhost:8000/pay.html --safety $s --turns 6 --tag g4_$s --task "이 페이지에서 무선 키보드를 구매하기 위해 결제하기 버튼을 누르세요." 2>&1 | grep -E "call:|SAFETY|->|finished|SUMMARY|RESULT|Error" | cut -c1-300; done